In [1]:
import numpy as np
import pandas as pd
from modules.bo import optimize_acqf_and_get_observation,BOLoop,initialize_gp_model,InitialSamples
from modules.msr import raasp_ic_generator,RAASP_sampler
from modules.sumo import create_initial_tl_df,check_feasibility,update_green_times,ErrorFunction,SumoSimulation,get_upper_bounds
import sumolib
from lxml import etree
import torch
import sympy as sp
from torch.quasirandom import SobolEngine
import matplotlib.pyplot as plt

c:\Users\sh032\anaconda3\envs\thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Creation of the Ground Truth Dataframe
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="Ingolstadt Network/InTAS_gt.sumocfg"
output_dir="gt_data/"

gt_sumo_simulator=SumoSimulation(num_replications,cfg_file,output_dir,tl_ids)

In [ ]:
# Creation of the Ground Truth Dataframe
epoch_index="gt"

gt_sumo_simulator.simulation(epoch_index)

In [8]:
df_gt=pd.read_csv("./gt_data/gt_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]

additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
output_dir="initial_output/"
additional_xml="./Ingolstadt Network/additional.xml"
dtype=torch.double

init_sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

init_objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=init_sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

initial_sampler=InitialSamples(dtype=dtype,
                               objective_function=init_objective_function,
                               initial_tl_df=initial_tl_df)

In [ ]:
sample_size=5
x_dim=98
lower_bounds=[5]*x_dim
upper_bounds=get_upper_bounds(initial_tl_df)
x_bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)
n_restart=5

init_X_norm_list=[]
init_Y_list=[]
init_df_list=[]

for i in range(n_restart):
    init_X_norm, init_Y, init_df=initial_sampler.get_initial_sample(sample_size,x_dim,x_bounds)
    init_X_norm_list.append(init_X_norm)
    init_Y_list.append(init_Y)
    init_df_list.append(init_df)

torch.save(init_X_norm_list,"./initial_samples/init_X_norm.pt")
torch.save(init_Y_list,"./initial_samples/init_Y.pt")
torch.save(init_df_list,"./initial_samples/init_df.pt")

# Vanilla BO

In [3]:
niter=20
num_restart=5
init_df_list=torch.load("./initial_samples/init_df.pt",weights_only=False)
output_dir="BO_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

raw_samples=128
batch_size=1
num_restarts_opti=5
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

BO_instance=BOLoop(niter=niter,
                   num_restart=num_restart,
                   init_df=init_df_list[0],
                   output_dir=output_dir,
                   bounds=bounds,
                   objective_function=objective_function,
                   dtype=dtype,
                   x_dim=x_dim,
                   device=device,
                   num_restarts_opti=num_restarts_opti,
                   raw_samples=raw_samples,
                   batch_size=batch_size,
                   initial_tl_df=initial_tl_df)

In [ ]:
BO_instance.run_loop(version="VANILLA",initial_length_scale_value=0.1,annotation="iss_5_ls_0.1",demand_scaling_factor=1)

# MSR Version

In [ ]:
niter=20
num_restart=5
init_df_list=torch.load("./initial_samples/init_df.pt",weights_only=False)
output_dir="BO_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

raw_samples=512
batch_size=1
num_restarts_opti=5
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))

raasp_sampler=RAASP_sampler(x_dim)

sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

BO_instance=BOLoop(niter=niter,
                   num_restart=num_restart,
                   init_df_list=init_df_list,
                   output_dir=output_dir,
                   bounds=bounds,
                   objective_function=objective_function,
                   dtype=dtype,
                   x_dim=x_dim,
                   device=device,
                   num_restarts_opti=num_restarts_opti,
                   raw_samples=raw_samples,
                   batch_size=batch_size,
                   initial_tl_df=initial_tl_df,
                   RAASP_sampler=raasp_sampler)

In [ ]:
BO_instance.run_loop(version="MSR",initial_length_scale_value=0.1*np.sqrt(98),annotation="iss_5_ls_0.1")

# Turbo with 1 trust regions and batch size = 1

In [5]:
n_trust_regions=1
x_dim=98
num_restarts=5
niter=20
tr_candidates_generator=TRCandidatesGeneration(x_dim)
init_sample_size=5


output_dir="turbo_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

batch_size=1
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)


trust_region_initial_df_list=torch.load("./initial_samples/init_df.pt",weights_only=False)
trust_region_initial_df_list=[[df] for df in trust_region_initial_df_list]

trust_region_initial_y_list=torch.load("./initial_samples/init_Y.pt",weights_only=False)
trust_region_initial_y_list=[[Y] for Y in trust_region_initial_y_list]

trust_region_initial_x_norm_list=torch.load("./initial_samples/init_X_norm.pt",weights_only=False)
trust_region_initial_x_norm_list=[[X] for X in trust_region_initial_x_norm_list]


torbo_bo_launcher=TurboLoop(
    n_trust_regions=n_trust_regions,
    dim=x_dim,
    num_restarts=num_restarts,
    tr_candidates_generator=tr_candidates_generator,
    objective_function=objective_function,
    trust_region_initial_x_norm_list=trust_region_initial_x_norm_list,
    trust_region_initial_y_list=trust_region_initial_y_list,
    trust_region_initial_df_list=trust_region_initial_df_list,
    batch_size=batch_size,
    niter=niter,
    bounds=bounds,
    output_dir=output_dir
)

In [ ]:
torbo_bo_launcher.turbo_loop(initial_length_scale_value=0.1,annotation="iss_5_ls_0.1")

# Comparison of algorithms

In [ ]:
def GP_plot(path,method_name):
    best_df_all=None
    for rs in range(5):
        if "TURBO" not in method_name:
            df=pd.read_csv(f"{path}{rs}.csv")
            #if method_name=="MSR":
                #df.loc[df["iter"]>0,"iter"]=df.loc[df["iter"]>0,"iter"]-1
        else:
            df=pd.read_csv(f"{path}{rs}_training.csv")

        best_df=df.groupby("iter").agg({"loss_estimate":"min"}).reset_index().rename(columns={"loss_estimate":"min_loss"})
        best_df["min_loss"]=best_df["min_loss"].cummin()
        if best_df_all is None:
            best_df_all=best_df
        else:
            best_df_all=pd.concat([best_df_all,best_df],axis=0,ignore_index=True)

    agg_best_df_all=best_df_all.groupby("iter").agg({"min_loss":["mean","std"]}).droplevel(0,axis=1).reset_index().rename(columns={"mean":"best_loss_mean","std":"best_loss_std"})
    plt.plot(agg_best_df_all.iter, agg_best_df_all.best_loss_mean, label=method_name,linewidth=2.2)
    plt.fill_between(agg_best_df_all.iter, 
                    agg_best_df_all.best_loss_mean - agg_best_df_all.best_loss_std, 
                    agg_best_df_all.best_loss_mean + agg_best_df_all.best_loss_std, 
                    alpha=0.12)

    print(f"df of {method_name} :")
    display(agg_best_df_all)

paths_list=[
    "./BO_output/BO_df_VANILLA_iss_5_ls_0.1_restart",
    "./BO_output/BO_df_MSR_iss_5_ls_0.1_restart",
    "./turbo_output/turbo_tr_0_iss_5_ls_0.1_rs_"
]

method_names_list=["VANILLA","MSR","TURBO-1"]

plt.figure()

for path,method_name in zip(paths_list,method_names_list):
    GP_plot(path,method_name)

plt.legend()

# Initial sample size = 15

In [4]:
df_gt=pd.read_csv("./gt_data/gt_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]

additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
output_dir="initial_output/"
additional_xml="./Ingolstadt Network/additional.xml"
dtype=torch.double

init_sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

init_objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=init_sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

initial_sampler=InitialSamples(dtype=dtype,
                               objective_function=init_objective_function,
                               initial_tl_df=initial_tl_df)

In [ ]:
sample_size=15
x_dim=98
lower_bounds=[5]*x_dim
upper_bounds=get_upper_bounds(initial_tl_df)
x_bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)
n_restart=3

init_X_norm_list=[]
init_Y_list=[]
init_df_list=[]

for i in range(n_restart):
    init_X_norm, init_Y, init_df=initial_sampler.get_initial_sample(sample_size,x_dim,x_bounds)
    init_X_norm_list.append(init_X_norm)
    init_Y_list.append(init_Y)
    init_df_list.append(init_df)

torch.save(init_X_norm_list,"./initial_samples/15_init_X_norm.pt")
torch.save(init_Y_list,"./initial_samples/15_init_Y.pt")
torch.save(init_df_list,"./initial_samples/15_init_df.pt")

# Vanilla BO

In [10]:
niter=20
num_restart=5
init_df_list=torch.load("./initial_samples/15_init_df.pt",weights_only=False)
output_dir="BO_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

raw_samples=128
batch_size=1
num_restarts_opti=5
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

BO_instance=BOLoop(niter=niter,
                   num_restart=num_restart,
                   init_df_list=init_df_list,
                   output_dir=output_dir,
                   bounds=bounds,
                   objective_function=objective_function,
                   dtype=dtype,
                   x_dim=x_dim,
                   device=device,
                   num_restarts_opti=num_restarts_opti,
                   raw_samples=raw_samples,
                   batch_size=batch_size,
                   initial_tl_df=initial_tl_df)

In [ ]:
BO_instance.run_loop(version="VANILLA",initial_length_scale_value=0.1,annotation="iss_15_ls_0.1")

# Vanilla BO with scaled lengthscale

In [11]:
niter=20
num_restart=5
init_df_list=torch.load("./initial_samples/15_init_df.pt",weights_only=False)
output_dir="BO_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

raw_samples=128
batch_size=1
num_restarts_opti=5
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

BO_instance=BOLoop(niter=niter,
                   num_restart=num_restart,
                   init_df_list=init_df_list,
                   output_dir=output_dir,
                   bounds=bounds,
                   objective_function=objective_function,
                   dtype=dtype,
                   x_dim=x_dim,
                   device=device,
                   num_restarts_opti=num_restarts_opti,
                   raw_samples=raw_samples,
                   batch_size=batch_size,
                   initial_tl_df=initial_tl_df)

In [ ]:
BO_instance.run_loop(version="VANILLA",initial_length_scale_value=0.1*np.sqrt(98),annotation="iss_15_ls_0.1scaled")

# MSR BO

In [16]:
niter=20
num_restart=5
init_df_list=torch.load("./initial_samples/15_init_df.pt",weights_only=False)
output_dir="BO_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

raw_samples=512
batch_size=1
num_restarts_opti=5
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))

raasp_sampler=RAASP_sampler(x_dim)

sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

BO_instance=BOLoop(niter=niter,
                   num_restart=num_restart,
                   init_df_list=init_df_list,
                   output_dir=output_dir,
                   bounds=bounds,
                   objective_function=objective_function,
                   dtype=dtype,
                   x_dim=x_dim,
                   device=device,
                   num_restarts_opti=num_restarts_opti,
                   raw_samples=raw_samples,
                   batch_size=batch_size,
                   initial_tl_df=initial_tl_df,
                   RAASP_sampler=raasp_sampler)

In [ ]:
BO_instance.run_loop(version="MSR",initial_length_scale_value=0.1*np.sqrt(98),annotation="iss_15_ls_0.1")

# TurBO 1

In [18]:
n_trust_regions=1
x_dim=98
num_restarts=5
niter=20
tr_candidates_generator=TRCandidatesGeneration(x_dim)
init_sample_size=5


output_dir="turbo_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

batch_size=1
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)


trust_region_initial_df_list=torch.load("./initial_samples/15_init_df.pt",weights_only=False)
trust_region_initial_df_list=[[df] for df in trust_region_initial_df_list]

trust_region_initial_y_list=torch.load("./initial_samples/15_init_Y.pt",weights_only=False)
trust_region_initial_y_list=[[Y] for Y in trust_region_initial_y_list]

trust_region_initial_x_norm_list=torch.load("./initial_samples/15_init_X_norm.pt",weights_only=False)
trust_region_initial_x_norm_list=[[X] for X in trust_region_initial_x_norm_list]


torbo_bo_launcher=TurboLoop(
    n_trust_regions=n_trust_regions,
    dim=x_dim,
    num_restarts=num_restarts,
    tr_candidates_generator=tr_candidates_generator,
    objective_function=objective_function,
    trust_region_initial_x_norm_list=trust_region_initial_x_norm_list,
    trust_region_initial_y_list=trust_region_initial_y_list,
    trust_region_initial_df_list=trust_region_initial_df_list,
    batch_size=batch_size,
    niter=niter,
    bounds=bounds,
    output_dir=output_dir
)

In [ ]:
torbo_bo_launcher.turbo_loop(initial_length_scale_value=0.1,annotation="iss_15_ls_0.1")

# TurBO 1 Scaled

In [14]:
n_trust_regions=1
x_dim=98
num_restarts=5
niter=20
tr_candidates_generator=TRCandidatesGeneration(x_dim)
init_sample_size=5


output_dir="turbo_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

batch_size=1
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)


trust_region_initial_df_list=torch.load("./initial_samples/15_init_df.pt",weights_only=False)
trust_region_initial_df_list=[[df] for df in trust_region_initial_df_list]

trust_region_initial_y_list=torch.load("./initial_samples/15_init_Y.pt",weights_only=False)
trust_region_initial_y_list=[[Y] for Y in trust_region_initial_y_list]

trust_region_initial_x_norm_list=torch.load("./initial_samples/15_init_X_norm.pt",weights_only=False)
trust_region_initial_x_norm_list=[[X] for X in trust_region_initial_x_norm_list]


torbo_bo_launcher=TurboLoop(
    n_trust_regions=n_trust_regions,
    dim=x_dim,
    num_restarts=num_restarts,
    tr_candidates_generator=tr_candidates_generator,
    objective_function=objective_function,
    trust_region_initial_x_norm_list=trust_region_initial_x_norm_list,
    trust_region_initial_y_list=trust_region_initial_y_list,
    trust_region_initial_df_list=trust_region_initial_df_list,
    batch_size=batch_size,
    niter=niter,
    bounds=bounds,
    output_dir=output_dir
)

In [ ]:
torbo_bo_launcher.turbo_loop(initial_length_scale_value=0.1*np.sqrt(98),annotation="iss_15_ls_0.1scaled")

# Turbo-m

In [36]:
n_trust_regions=3
x_dim=98
num_restarts=5
niter=20
tr_candidates_generator=TRCandidatesGeneration(x_dim)
init_sample_size=5


output_dir="turbo_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

batch_size=1
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)


initial_df_list=torch.load("./initial_samples/15_init_df.pt",weights_only=False)
trust_region_initial_df_list=[[df[i:i+5] for i in range(0,len(df),5)] for df in initial_df_list]

initial_y_list=torch.load("./initial_samples/15_init_Y.pt",weights_only=False)
trust_region_initial_y_list=[[df[i:i+5] for i in range(0,len(df),5)] for df in initial_y_list]

initial_x_norm_list=torch.load("./initial_samples/15_init_X_norm.pt",weights_only=False)
trust_region_initial_x_norm_list=[[df[i:i+5] for i in range(0,len(df),5)] for df in initial_x_norm_list]


torbo_bo_launcher=TurboLoop(
    n_trust_regions=n_trust_regions,
    dim=x_dim,
    num_restarts=num_restarts,
    tr_candidates_generator=tr_candidates_generator,
    objective_function=objective_function,
    trust_region_initial_x_norm_list=trust_region_initial_x_norm_list,
    trust_region_initial_y_list=trust_region_initial_y_list,
    trust_region_initial_df_list=trust_region_initial_df_list,
    batch_size=batch_size,
    niter=niter,
    bounds=bounds,
    output_dir=output_dir
)

In [ ]:
torbo_bo_launcher.turbo_loop(initial_length_scale_value=0.1,annotation="iss_15_ls_0.1_tr3_q1")

# Comparison of algorithms

In [ ]:
def GP_plot(path,method_name):
    best_df_all=None
    for rs in range(5):
        if "TURBO" not in method_name:
            df=pd.read_csv(f"{path}{rs}.csv")
        elif method_name=="TURBO-1" or method_name=="TURBO-1 SCALED":
            df=pd.read_csv(f"{path}{rs}_training.csv")
        else:
            df=None
            for turbo_m_path in path:
                turbo_m_df=pd.read_csv(f"{turbo_m_path}{rs}_training.csv")
                if df is None:
                    df=turbo_m_df
                else:
                    df=pd.concat([df,turbo_m_df])

        best_df=df.groupby("iter").agg({"loss_estimate":"min"}).reset_index().rename(columns={"loss_estimate":"min_loss"})
        best_df["min_loss"]=best_df["min_loss"].cummin()
        if best_df_all is None:
            best_df_all=best_df
        else:
            best_df_all=pd.concat([best_df_all,best_df],axis=0,ignore_index=True)
        

    agg_best_df_all=best_df_all.groupby("iter").agg({"min_loss":["mean","std"]}).droplevel(0,axis=1).reset_index().rename(columns={"mean":"best_loss_mean","std":"best_loss_std"})
    plt.plot(agg_best_df_all.iter, agg_best_df_all.best_loss_mean, label=method_name,linewidth=2.2)
    plt.fill_between(agg_best_df_all.iter, 
                    agg_best_df_all.best_loss_mean - agg_best_df_all.best_loss_std, 
                    agg_best_df_all.best_loss_mean + agg_best_df_all.best_loss_std, 
                    alpha=0.12)

    print(f"df of {method_name} :")
    display(agg_best_df_all)

paths_list=[
    "./BO_output/BO_df_VANILLA_iss_15_ls_0.1_restart",
    "./BO_output/BO_df_VANILLA_iss_15_ls_0.1scaled_restart",
    "./BO_output/BO_df_MSR_iss_15_ls_0.1_restart",
    "./turbo_output/turbo_tr_0_iss_15_ls_0.1_rs_",
    "./turbo_output/turbo_tr_0_iss_15_ls_0.1scaled_rs_",
    ["./turbo_output/turbo_tr_0_iss_15_ls_0.1_tr3_q1_rs_",
    "./turbo_output/turbo_tr_1_iss_15_ls_0.1_tr3_q1_rs_",
    "./turbo_output/turbo_tr_2_iss_15_ls_0.1_tr3_q1_rs_"]
]

method_names_list=["VANILLA","VANILLA SCALED","MSR","TURBO-1","TURBO-1 SCALED","TURBO-M"]

plt.figure()

for path,method_name in zip(paths_list,method_names_list):
    GP_plot(path,method_name)

plt.legend()

# We try with different level of demand

In [5]:
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
num_replications=5

cfg_file_gt="Ingolstadt Network/InTAS_gt.sumocfg"
output_dir_gt="gt_data/"
gt_sumo_simulator=SumoSimulation(num_replications,cfg_file_gt,output_dir_gt,tl_ids)



gt_sumo_simulator.simulation(epoch_index="gt")
df_gt=pd.read_csv(f"./gt_data/gt_InTAS.travel_times.csv")

In [4]:
df_gt=pd.read_csv(f"./gt_data/gt_InTAS.travel_times.csv")
g_vec=np.full(shape=(98,),fill_value=33)


net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
num_replications=3

output_dir_init="initial_output/"


cfg_file="./Ingolstadt Network/InTAS.sumocfg"
output_dir_init="initial_output/"
additional_xml="./Ingolstadt Network/additional.xml"
dtype=torch.double

init_sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir_init,
                                   tl_ids=tl_ids,
                                   )


error_function=ErrorFunction(
        initial_tl_df=initial_tl_df,
        output_dir=output_dir_init,
        sumo_simulator=init_sumo_simulator,
        df_gt=df_gt,
        additional_xml=additional_xml,
        tl_ids=tl_ids
    )


error_function.compute_error(g_vec=g_vec,epoch_index="test",demand_scaling_factor=3)

G VEC ALL :
[ 36  33  33  19  33  33  33  51  33  33  33  54  36  33  51  33  33  42
  27  33  33  25  33  38  51  33  33  33  51  33  33  51  33   3  33  51
  33  41  33  51  33   3  33  51  47  33  33  42  33  39  33  42  33  29
  21  33  51  33  33  21  42  33  33  46  33  32  51  33  33  51  42  33
  42  33  33  51  33  42  33  42   2  33  33   3  42  33  33  21  42  33
  42  33  33  39  33  39  33  51  33  36  33  51  42  33  33  51  33  33
  42  33  33  42  25  33  33  42  33  15  33  25  33  31  21  33  15  33
  33  54  33   7  33  42  33  42  33  42  33  21  33  11  33  25  33  33
  33  33  33  18  33  42  33  42  33  33  33  42  42  33  33  33  33  15
  33  28  33 -12  33  42  33  26  39  33  42  33  33  33  42  33  15  33
  33  42  33  51  33  36  33  25  33  42  33  36  33  42  33  42]
Wrote ./Ingolstadt Network/additional.xml


C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)


np.float64(2281949.4633728852)

# Demand scaling factor = 3

In [ ]:
#demand_scaling_factors_list=[1,1.5,2,2.5,3]


net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
num_replications=5

cfg_file_gt="Ingolstadt Network/InTAS_gt.sumocfg"
output_dir_gt="gt_data/"
gt_sumo_simulator=SumoSimulation(num_replications,cfg_file_gt,output_dir_gt,tl_ids)

cfg_file="./Ingolstadt Network/InTAS.sumocfg"
output_dir_init="initial_output/"
additional_xml="./Ingolstadt Network/additional.xml"
dtype=torch.double

init_sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir_init,
                                   tl_ids=tl_ids,
                                   )


sample_size=15
x_dim=98
lower_bounds=[5]*x_dim
upper_bounds=get_upper_bounds(initial_tl_df)
x_bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)

n_restart=3

demand_scaling_factor=3
epoch_index=f"gt_dsf_{demand_scaling_factor}"
gt_sumo_simulator.simulation(epoch_index,demand_scaling_factor=demand_scaling_factor)
df_gt=pd.read_csv(f"./gt_data/gt_dsf_{demand_scaling_factor}_InTAS.travel_times.csv")

init_objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir_init,
    sumo_simulator=init_sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

initial_sampler=InitialSamples(dtype=dtype,
                            objective_function=init_objective_function,
                            initial_tl_df=initial_tl_df)


init_X_norm_list=[]
init_Y_list=[]
init_df_list=[]

for i in range(n_restart):
    init_X_norm, init_Y, init_df=initial_sampler.get_initial_sample(sample_size,x_dim,x_bounds,demand_scaling_factor)
    init_X_norm_list.append(init_X_norm)
    init_Y_list.append(init_Y)
    init_df_list.append(init_df)

torch.save(init_X_norm_list,f"./initial_samples/init_X_norm_dsf_{demand_scaling_factor}.pt")
torch.save(init_Y_list,f"./initial_samples/init_Y_dsf_{demand_scaling_factor}.pt")
torch.save(init_df_list,f"./initial_samples/init_df_dsf_{demand_scaling_factor}.pt")

# Vanilla method

In [2]:
niter=10
num_restart=3
init_df_list=torch.load("./initial_samples/init_df_dsf_3.pt",weights_only=False)
output_dir="BO_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_dsf_3_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

raw_samples=min(100*x_dim,5000)
batch_size=1
num_restarts_opti=5
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

BO_instance=BOLoop(niter=niter,
                   num_restart=num_restart,
                   init_df=init_df_list[0],
                   output_dir=output_dir,
                   bounds=bounds,
                   objective_function=objective_function,
                   dtype=dtype,
                   x_dim=x_dim,
                   device=device,
                   num_restarts_opti=num_restarts_opti,
                   raw_samples=raw_samples,
                   batch_size=batch_size,
                   initial_tl_df=initial_tl_df,
                   )

In [ ]:
BO_instance.run_loop(version="VANILLA",initial_length_scale_value=0.1,annotation="iss_15_ls_0.1_dsf_3",demand_scaling_factor=3)

# Vanilla Scaled method

In [4]:
niter=10
num_restart=3
init_df_list=torch.load("./initial_samples/init_df_dsf_3.pt",weights_only=False)
output_dir="BO_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_dsf_3_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

raw_samples=min(100*x_dim,5000)
batch_size=1
num_restarts_opti=5
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

BO_instance=BOLoop(niter=niter,
                   num_restart=num_restart,
                   init_df=init_df_list[0],
                   output_dir=output_dir,
                   bounds=bounds,
                   objective_function=objective_function,
                   dtype=dtype,
                   x_dim=x_dim,
                   device=device,
                   num_restarts_opti=num_restarts_opti,
                   raw_samples=raw_samples,
                   batch_size=batch_size,
                   initial_tl_df=initial_tl_df,
                   )

In [ ]:
BO_instance.run_loop(version="VANILLA",initial_length_scale_value=0.1*np.sqrt(x_dim),annotation="iss_15_ls_0.1scaled_dsf_3",demand_scaling_factor=3)

# Vanilla default lengthscale

In [6]:
niter=10
num_restart=3
init_df_list=torch.load("./initial_samples/init_df_dsf_3.pt",weights_only=False)
output_dir="BO_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_dsf_3_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

raw_samples=min(100*x_dim,5000)
batch_size=1
num_restarts_opti=5
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

BO_instance=BOLoop(niter=niter,
                   num_restart=num_restart,
                   init_df=init_df_list[0],
                   output_dir=output_dir,
                   bounds=bounds,
                   objective_function=objective_function,
                   dtype=dtype,
                   x_dim=x_dim,
                   device=device,
                   num_restarts_opti=num_restarts_opti,
                   raw_samples=raw_samples,
                   batch_size=batch_size,
                   initial_tl_df=initial_tl_df,
                   )

In [ ]:
BO_instance.run_loop(version="VANILLA",initial_length_scale_value=0.6931,annotation="iss_15_ls_0.6931_dsf_3",demand_scaling_factor=3)

# MSR

In [11]:
niter=10
num_restart=3
init_df_list=torch.load("./initial_samples/init_df_dsf_3.pt",weights_only=False)
output_dir="BO_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_dsf_3_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

raw_samples=min(100*x_dim,5000)
batch_size=1
num_restarts_opti=5
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))

raasp_sampler=RAASP_sampler(x_dim)

sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)

BO_instance=BOLoop(niter=niter,
                   num_restart=num_restart,
                   init_df=init_df_list[0],
                   output_dir=output_dir,
                   bounds=bounds,
                   objective_function=objective_function,
                   dtype=dtype,
                   x_dim=x_dim,
                   device=device,
                   num_restarts_opti=num_restarts_opti,
                   raw_samples=raw_samples,
                   batch_size=batch_size,
                   initial_tl_df=initial_tl_df,
                   RAASP_sampler=raasp_sampler)

In [ ]:
BO_instance.run_loop(version="MSR",initial_length_scale_value=0.1*np.sqrt(x_dim),annotation="iss_15_ls_0.1scaled_dsf_3",demand_scaling_factor=3)

# Turbo-1 default lengthscale

In [26]:
n_trust_regions=1
x_dim=98
num_restarts=3
niter=10
tr_candidates_generator=TRCandidatesGeneration(x_dim)
init_sample_size=15
num_restarts_opti=5


output_dir="turbo_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_dsf_3_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

batch_size=1
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)


trust_region_initial_df_list=torch.load("./initial_samples/init_df_dsf_3.pt",weights_only=False)
trust_region_initial_df_list=[trust_region_initial_df_list[0]]

trust_region_initial_y_list=torch.load(f"./initial_samples/init_Y_dsf_3.pt",weights_only=False)
trust_region_initial_y_list=[trust_region_initial_y_list[0]]

trust_region_initial_x_norm_list=torch.load("./initial_samples/init_X_norm_dsf_3.pt",weights_only=False)
trust_region_initial_x_norm_list=[trust_region_initial_x_norm_list[0]]


torbo_bo_launcher=TurboLoop(
    n_trust_regions=n_trust_regions,
    dim=x_dim,
    num_restarts=num_restarts,
    tr_candidates_generator=tr_candidates_generator,
    objective_function=objective_function,
    trust_region_initial_x_norm_list=trust_region_initial_x_norm_list,
    trust_region_initial_y_list=trust_region_initial_y_list,
    trust_region_initial_df_list=trust_region_initial_df_list,
    batch_size=batch_size,
    niter=niter,
    bounds=bounds,
    output_dir=output_dir
)

In [ ]:
torbo_bo_launcher.turbo_loop(initial_length_scale_value=0.6931,annotation="iss_15_ls_0.6931_dsf_3",demand_scaling_factor=3)

# Turbo-1 

In [29]:
n_trust_regions=1
x_dim=98
num_restarts=3
niter=10
tr_candidates_generator=TRCandidatesGeneration(x_dim)
init_sample_size=15
num_restarts_opti=5


output_dir="turbo_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_dsf_3_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

batch_size=1
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)


trust_region_initial_df_list=torch.load("./initial_samples/init_df_dsf_3.pt",weights_only=False)
trust_region_initial_df_list=[trust_region_initial_df_list[0]]

trust_region_initial_y_list=torch.load(f"./initial_samples/init_Y_dsf_3.pt",weights_only=False)
trust_region_initial_y_list=[trust_region_initial_y_list[0]]

trust_region_initial_x_norm_list=torch.load("./initial_samples/init_X_norm_dsf_3.pt",weights_only=False)
trust_region_initial_x_norm_list=[trust_region_initial_x_norm_list[0]]


torbo_bo_launcher=TurboLoop(
    n_trust_regions=n_trust_regions,
    dim=x_dim,
    num_restarts=num_restarts,
    tr_candidates_generator=tr_candidates_generator,
    objective_function=objective_function,
    trust_region_initial_x_norm_list=trust_region_initial_x_norm_list,
    trust_region_initial_y_list=trust_region_initial_y_list,
    trust_region_initial_df_list=trust_region_initial_df_list,
    batch_size=batch_size,
    niter=niter,
    bounds=bounds,
    output_dir=output_dir
)

In [ ]:
torbo_bo_launcher.turbo_loop(initial_length_scale_value=0.1,annotation="iss_15_ls_0.1_dsf_3",demand_scaling_factor=3)

# Turbo-1 lengthscale scaled

In [3]:
n_trust_regions=1
x_dim=98
num_restarts=3
niter=10
tr_candidates_generator=TRCandidatesGeneration(x_dim)
init_sample_size=15
num_restarts_opti=5


output_dir="turbo_output/"
dtype=torch.double
x_dim=98
lower_bounds=[5]*x_dim
additional_xml_gt="./Ingolstadt Network/additional_gt.xml"
initial_tl_df=create_initial_tl_df(additional_xml_gt)
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=torch.tensor(
    [lower_bounds,
    upper_bounds],
    dtype=dtype
)


df_gt=pd.read_csv("./gt_data/gt_dsf_3_InTAS.travel_times.csv")
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=5
cfg_file="./Ingolstadt Network/InTAS.sumocfg"
additional_xml="./Ingolstadt Network/additional.xml"

batch_size=1
device=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))


sumo_simulator=SumoSimulation(num_replications=num_replications,
                                   cfg_file=cfg_file,
                                   output_dir=output_dir,
                                   tl_ids=tl_ids,
                                   )

objective_function=ErrorFunction(
    initial_tl_df=initial_tl_df,
    output_dir=output_dir,
    sumo_simulator=sumo_simulator,
    df_gt=df_gt,
    additional_xml=additional_xml,
    tl_ids=tl_ids
)


trust_region_initial_df_list=torch.load("./initial_samples/init_df_dsf_3.pt",weights_only=False)
trust_region_initial_df_list=[trust_region_initial_df_list[0]]

trust_region_initial_y_list=torch.load(f"./initial_samples/init_Y_dsf_3.pt",weights_only=False)
trust_region_initial_y_list=[trust_region_initial_y_list[0]]

trust_region_initial_x_norm_list=torch.load("./initial_samples/init_X_norm_dsf_3.pt",weights_only=False)
trust_region_initial_x_norm_list=[trust_region_initial_x_norm_list[0]]


torbo_bo_launcher=TurboLoop(
    n_trust_regions=n_trust_regions,
    dim=x_dim,
    num_restarts=num_restarts,
    tr_candidates_generator=tr_candidates_generator,
    objective_function=objective_function,
    trust_region_initial_x_norm_list=trust_region_initial_x_norm_list,
    trust_region_initial_y_list=trust_region_initial_y_list,
    trust_region_initial_df_list=trust_region_initial_df_list,
    batch_size=batch_size,
    niter=niter,
    bounds=bounds,
    output_dir=output_dir
)

In [ ]:
torbo_bo_launcher.turbo_loop(initial_length_scale_value=0.1*np.sqrt(98),annotation="iss_15_ls_0.1scaled_dsf_3",demand_scaling_factor=3)

# Comparison of all methods

In [ ]:
def GP_plot(path,method_name):
    best_df_all=None
    for rs in range(3):
        if "TURBO" not in method_name:
            df=pd.read_csv(f"{path}{rs}.csv")
        elif "TURBO-1" in method_name:
            df=pd.read_csv(f"{path}{rs}_training.csv")
        else:
            df=None
            for turbo_m_path in path:
                turbo_m_df=pd.read_csv(f"{turbo_m_path}{rs}_training.csv")
                if df is None:
                    df=turbo_m_df
                else:
                    df=pd.concat([df,turbo_m_df])

        best_df=df.groupby("iter").agg({"loss_estimate":"min"}).reset_index().rename(columns={"loss_estimate":"min_loss"})
        best_df["min_loss"]=best_df["min_loss"].cummin()
        if best_df_all is None:
            best_df_all=best_df
        else:
            best_df_all=pd.concat([best_df_all,best_df],axis=0,ignore_index=True)
        

    agg_best_df_all=best_df_all.groupby("iter").agg({"min_loss":["mean","std"]}).droplevel(0,axis=1).reset_index().rename(columns={"mean":"best_loss_mean","std":"best_loss_std"})
    plt.plot(agg_best_df_all.iter, agg_best_df_all.best_loss_mean, label=method_name,linewidth=2.2)
    plt.fill_between(agg_best_df_all.iter, 
                    agg_best_df_all.best_loss_mean - agg_best_df_all.best_loss_std, 
                    agg_best_df_all.best_loss_mean + agg_best_df_all.best_loss_std, 
                    alpha=0.12)

    print(f"df of {method_name} :")
    display(agg_best_df_all)

paths_list=[
    "./BO_output/BO_df_VANILLA_iss_15_ls_0.1_dsf_3_restart",
    "./BO_output/BO_df_VANILLA_iss_15_ls_0.1scaled_dsf_3_restart",
    "./BO_output/BO_df_VANILLA_iss_15_ls_0.6931_dsf_3_restart",
    "./BO_output/BO_df_MSR_iss_15_ls_0.1scaled_dsf_3_restart",
    "./turbo_output/turbo_tr_0_iss_15_ls_0.1_dsf_3_rs_",
    "./turbo_output/turbo_tr_0_iss_15_ls_0.1scaled_dsf_3_rs_",
    "./turbo_output/turbo_tr_0_iss_15_ls_0.6931_dsf_3_rs_",
]

method_names_list=["VANILLA","VANILLA SCALED","VANILLA DEFAULT","MSR","TURBO-1","TURBO-1 SCALED","TURBO-1 DEFAULT"]

plt.figure()

for path,method_name in zip(paths_list,method_names_list):
    GP_plot(path,method_name)

plt.legend()

In [ ]:
net=sumolib.net.readNet("Ingolstadt Network/ingolstadt.net.xml")
tl_ids=[tl.getID() for tl in net.getTrafficLights()]
num_replications=3
cfg_file="Ingolstadt Network/InTAS_gt.sumocfg"
output_dir="gt_data/"

gt_sumo_simulator=SumoSimulation(num_replications,cfg_file,output_dir,tl_ids)
epoch_index="gt"
gt_sumo_simulator.simulation(epoch_index)

C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
